In [0]:
pudl_path = "s3a://pudl.catalyst.coop/stable/"

In [0]:
files = dbutils.fs.ls(pudl_path)
for f in files:
    print(f"{f.name}")

####FERC O&M table

In [0]:
# Load the FERC steam plants files             
ferc_steam_onm = spark.read.parquet(pudl_path +"core_ferc1__yearly_steam_plants_sched402.parquet")

# Create a view
ferc_steam_onm.createOrReplaceTempView("ferc_steam_onm_first")

# Reading O&M file
print("O&M Rows:", ferc_steam_onm.count())
print("O&M Columns:", len(ferc_steam_onm.columns))
ferc_steam_onm.printSchema()
display(ferc_steam_onm.limit(5))


####FERC fuel table

In [0]:
# fuel file (the amount of fuel burned)
ferc_steam_fuel = spark.read.parquet(pudl_path +"core_ferc1__yearly_steam_plants_fuel_sched402.parquet")

# Create a view
ferc_steam_fuel.createOrReplaceTempView("ferc_steam_fuel_first")


# Rading Fuel file
print("Fuel Rows:", ferc_steam_fuel.count())
print("Fuel Columns:", len(ferc_steam_fuel.columns))
ferc_steam_fuel.printSchema()   
display(ferc_steam_fuel.limit(5))

####EIA-860 plant inventory

In [0]:
# Load EIA-860 plant inventory
eia_plants = spark.read.parquet(pudl_path + "core_eia860__scd_plants.parquet")

# Create a view
eia_plants.createOrReplaceTempView("eia_plants_first")


print("EIA-860 rows:", eia_plants.count())
eia_plants.printSchema()
display(eia_plants.limit(5))

#### EIA-923 generation data

In [0]:
# Load EIA-923 monthly generation
eia_gen = spark.read.parquet(pudl_path + "core_eia923__monthly_generation_fuel.parquet")


# Create a view
eia_gen.createOrReplaceTempView("eia_gen_first")

print("EIA-923 rows:", eia_gen.count())
eia_gen.printSchema()
display(eia_gen.limit(5))


##Joining FERC Form 1 IDs to EIA IDs

An Irregularity found out by PUDL is between FERC IDs and EIA IDs, They have many unalgined keys hence they cannot be joined efficiently without extensive data preprocessing. Luckily, PUDL has manually created [association tables](https://docs.catalyst.coop/pudl/en/latest/data_sources/ferc1.html#notable-irregularities) between them. Using these association tables, it becomes very easy to join these  tables using the following Association tables:


In [0]:
# Load FERC → PUDL utility crosswalk
ferc_pudl = spark.read.parquet(pudl_path + "core_pudl__assn_ferc1_pudl_utilities.parquet")

# Create a view
ferc_pudl.createOrReplaceTempView("ferc_pudl_first")

# Load EIA → PUDL utility crosswalk
eia_pudl = spark.read.parquet(pudl_path + "core_pudl__assn_eia_pudl_utilities.parquet")

# Create a view
eia_pudl.createOrReplaceTempView("eia_pudl_first")

# Confirm schema — utility_id_pudl must appear in both
print("FERC-PUDL")
print("Rows:", ferc_pudl.count())
ferc_pudl.printSchema()
display(ferc_pudl.limit(5))

print("EIA-PUDL")
print("Rows:", eia_pudl.count())
eia_pudl.printSchema()
display(eia_pudl.limit(5))

#  FERC plant name → PUDL plant ID
ferc_pudl_plants_first = spark.read.parquet(
    pudl_path + "core_pudl__assn_ferc1_pudl_plants.parquet"
)

ferc_pudl_plants_first.createOrReplaceTempView("ferc_pudl_plants_first")

print("ferc_pudl_plants_first")
print("  rows:", ferc_pudl_plants_first.count())
ferc_pudl_plants_first.printSchema()
display(ferc_pudl_plants_first.limit(10))



#### Generators Data
To understand if a plant is already out of operation (partially or completely) we need generator level data, a plant can have multiple Generators producing energy. It can be that some of the generators are out of operation and some are still generating.

In [0]:
# Pre-joined output table for generators 
#  Generators Data is needed to understand with of them are still operating the heirarchy is : Utility(Entity) -> Plant -> Generator
eia_generators = spark.read.parquet(pudl_path + "out_eia__yearly_generators.parquet")

# Creating a View
eia_generators.createOrReplaceTempView("eia_generators_first")

print("Rows:", eia_generators.count())
eia_generators.printSchema()
display(eia_generators.limit(5))

In [0]:
# Plant-level entity table
eia_plants_scd = spark.read.parquet(pudl_path + "core_eia860__scd_plants.parquet")

# Create View
eia_plants_scd.createOrReplaceTempView("eia_plants_scd_first")

eia_plants_scd.printSchema()
display(eia_plants_scd.limit(5))

In [0]:
#  Looking for different sectors
from pyspark.sql import functions as F
display(
    eia_plants_scd
    .groupBy("sector_name_eia")
    .count()
    .orderBy(F.desc("count"))
)

#### Time Period of the Data

In [0]:
print(" FERC O&M ")
ferc_steam_onm.select(F.min("report_year"), F.max("report_year")).show()

print(" FERC Fuel ")
ferc_steam_fuel.select(F.min("report_year"), F.max("report_year")).show()

print(" EIA generators ")
eia_gen.select(F.min("report_date"), F.max("report_date")).show()

print(" EIA plants SCD ")
eia_plants_scd.select(F.min("report_date"), F.max("report_date")).show()

In [0]:
# Check for Duplicate Power plan namtes 
# Group by to spot duplicates
plant_name_variants = (
    ferc_steam_onm
    .filter(F.col("report_year") >= 2015)
    .select(F.lower(F.col("plant_name_ferc1")), "plant_name_ferc1")
    .groupBy("plant_name_ferc1")
    .agg(F.collect_set("plant_name_ferc1").alias("variants"))
    .filter(F.size("variants") > 1)
    .orderBy(F.desc(F.size("variants")))
)

display(plant_name_variants.limit(30))

In [0]:
# Different Kind of Fuel used in Power Plants
display(
    spark.sql("""
        SELECT fuel_type_code_pudl, COUNT(*) AS rows
        FROM ferc_steam_fuel_first
        WHERE report_year >= 2015
        GROUP BY fuel_type_code_pudl
        ORDER BY rows DESC
        LIMIT 20
    """)
)

# Plants with multiple fuel types in a random year between 2015 and 2025
import random
random_year = random.randint(2015, 2025)

display(
    spark.sql(f"""
        SELECT report_year, plant_name_ferc1, COUNT(DISTINCT fuel_type_code_pudl) AS fuel_types, 
               COLLECT_SET(fuel_type_code_pudl) AS fuel_type_codes
        FROM ferc_steam_fuel_first
        WHERE report_year = {random_year}
        GROUP BY report_year, plant_name_ferc1
        HAVING fuel_types > 1
        ORDER BY fuel_types DESC
        LIMIT 20
    """)
)

# The same plant occurs more than in quite some years. 
# Bad for aggregation as different fuels uses different measuring units

#### Initial findings
This Query emphasis on the sheer quantity of plants running solely on fossil fuel. The task moving forward would be to figure out how many of these plants are not viable anymore.

In [0]:
#  how many FERC utilities have a duplicate PUDL ID?
unmatched = spark.sql("""
    SELECT COUNT(DISTINCT fo.utility_id_ferc1) AS unmatched_ferc_utilities
    FROM ferc_steam_onm_first fo
    LEFT JOIN ferc_pudl_first cw
        ON fo.utility_id_ferc1 = cw.utility_id_ferc1
    WHERE cw.utility_id_pudl IS NULL
      AND fo.report_year >= 2015
""")
unmatched.show()


# Perfectly Normalised (Thank you PUDL :))

In [0]:
# Testing if the join works using the middle table eia_pudl_first
join_test = spark.sql("""
    SELECT
        fo.report_year,
        COUNT(DISTINCT fo.utility_id_ferc1) AS ferc_utilities,
        COUNT(DISTINCT eg.plant_id_eia)    AS eia_plants_reachable,
        COUNT(DISTINCT eg.generator_id)    AS eia_generators_reachable
    FROM ferc_steam_onm_first fo
    JOIN ferc_pudl_first cw  ON fo.utility_id_ferc1 = cw.utility_id_ferc1
    JOIN eia_pudl_first ep   ON cw.utility_id_pudl = ep.utility_id_pudl
    JOIN eia_generators_first eg
        ON ep.utility_id_eia = eg.utility_id_eia
        AND fo.report_year = YEAR(eg.report_date)
    WHERE fo.report_year >= 2015
    GROUP BY fo.report_year
    ORDER BY fo.report_year
""")
display(join_test)

#### Types of Operational status    

In [0]:
display(
    spark.sql("""
        SELECT
            operational_status,
            COUNT(*) AS generator_rows,
            COUNT(DISTINCT plant_id_eia) AS distinct_plants
        FROM eia_generators_first
        WHERE report_date > '2015-01-01'
        GROUP BY operational_status
        ORDER BY generator_rows DESC
    """)
)